# 02. Training Experiments - Person-based protocol

Notebook nay train va chon model cho Smart Posture Monitor theo split person-based.

Nguyen tac chinh:
- Chi load va validate dataset qua `src.preprocessing.prepare_dataset()`.
- Khong reimplement cleaning, imputation, dropna/fillna, schema validation, hay feature list.
- Holdout test persons chi dung de luu split manifest va kiem tra leakage.
- Tat ca baseline, cross-validation, hyperparameter tuning va model selection chi dung training persons.
- Final unseen-person evaluation se duoc lam rieng trong `evaluate.py`.


In [1]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from sklearn.model_selection import (
    GroupShuffleSplit,
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedGroupKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Make the notebook runnable from either project root or notebooks/.
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError("Khong tim thay project root chua thu muc src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import EXPECTED_CLASSES, ID_TO_LABEL, LABEL_TO_ID, prepare_dataset

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "features.csv"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TEST_PERSON_IDS = None  # Example: ["person03", "person07"]
N_SPLITS = 5
USE_XGBOOST_GPU = False
XGB_N_ITER = 20

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"Random state: {RANDOM_STATE}")
print(f"Holdout test size: {TEST_SIZE}")
print(f"Manual test persons: {TEST_PERSON_IDS}")


Project root: G:\PTIT TL\K1N3\PYTHON\BTL\smart_posture_monitor
Dataset path: G:\PTIT TL\K1N3\PYTHON\BTL\smart_posture_monitor\data\processed\features.csv
Random state: 42
Holdout test size: 0.2
Manual test persons: None


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    if isinstance(value, (Path,)):
        return str(value)
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(
        json.dumps(json_safe(payload), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def encoded_class_counts(y_values: pd.Series) -> dict[str, int]:
    counts = y_values.value_counts().sort_index()
    return {
        ID_TO_LABEL[int(class_id)]: int(counts.get(class_id, 0))
        for class_id in sorted(ID_TO_LABEL)
    }


def assert_all_classes_present(name: str, y_values: pd.Series) -> None:
    present = set(int(v) for v in pd.Series(y_values).unique())
    expected = set(ID_TO_LABEL.keys())
    missing = sorted(expected - present)
    if missing:
        missing_labels = [ID_TO_LABEL[class_id] for class_id in missing]
        raise ValueError(f"{name} bi thieu class: {missing_labels}")


def print_table(title: str, frame: pd.DataFrame) -> None:
    print("\n" + title)
    print("-" * len(title))
    print(frame.to_string(index=False))


In [3]:
prepared = prepare_dataset(DATA_PATH)
dataset_sha256 = sha256_file(DATA_PATH)

X = prepared.X.copy()
y = prepared.y.copy()
groups = prepared.groups.copy()
metadata = prepared.metadata.copy()
feature_columns = list(prepared.feature_columns)

print("Dataset da duoc load bang prepare_dataset().")
print(f"Samples: {len(prepared.df)}")
print(f"Features: {X.shape[1]}")
print(f"Persons: {groups.nunique()}")
print(f"Recordings: {metadata['recording_id'].nunique()}")
print(f"Classes: {len(EXPECTED_CLASSES)}")
print(f"Dataset SHA256: {dataset_sha256}")
print_table("Class distribution", prepared.df["label"].value_counts().reindex(EXPECTED_CLASSES, fill_value=0).rename_axis("label").reset_index(name="samples"))
print_table("Samples per person", metadata["person_id"].value_counts().sort_index().rename_axis("person_id").reset_index(name="samples"))


Dataset da duoc load bang prepare_dataset().
Samples: 4014
Features: 29
Persons: 14
Recordings: 16
Classes: 4
Dataset SHA256: 2bc189b67d6ac75dc8a95dfb487c743b16af0f99172eebf28ce725b6c5abbb62

Class distribution
------------------
         label  samples
       correct     1051
forward_slouch      889
     lean_left     1054
    lean_right     1020

Samples per person
------------------
person_id  samples
 person01      148
 person02      277
 person03      201
 person04      275
 person05      260
 person06      240
 person07      796
 person08      226
 person09      273
 person10      307
 person11      258
 person12      252
 person13      255
 person14      246


In [4]:
all_persons = np.array(sorted(groups.unique()))

if TEST_PERSON_IDS is None:
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))
else:
    requested_test_persons = set(str(person_id) for person_id in TEST_PERSON_IDS)
    unknown_persons = sorted(requested_test_persons - set(all_persons))
    if unknown_persons:
        raise ValueError(f"TEST_PERSON_IDS khong ton tai trong dataset: {unknown_persons}")
    test_mask = groups.isin(requested_test_persons).to_numpy()
    train_idx = np.where(~test_mask)[0]
    test_idx = np.where(test_mask)[0]

X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()
groups_train = groups.iloc[train_idx].copy()
metadata_train = metadata.iloc[train_idx].copy()

X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_test = groups.iloc[test_idx].copy()
metadata_test = metadata.iloc[test_idx].copy()

train_persons = sorted(groups_train.unique())
test_persons = sorted(groups_test.unique())
train_recording_ids = sorted(metadata_train["recording_id"].unique())
test_recording_ids = sorted(metadata_test["recording_id"].unique())

person_overlap = sorted(set(train_persons) & set(test_persons))
recording_overlap = sorted(set(train_recording_ids) & set(test_recording_ids))
if person_overlap:
    raise RuntimeError(f"Person leakage giua train/test: {person_overlap}")
if recording_overlap:
    raise RuntimeError(f"Recording leakage giua train/test: {recording_overlap}")

assert_all_classes_present("Train split", y_train)
assert_all_classes_present("Test split", y_test)

split_manifest = {
    "split_strategy": "manual_person_holdout" if TEST_PERSON_IDS is not None else "GroupShuffleSplit person holdout",
    "group_column": "person_id",
    "recording_column": "recording_id",
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "test_person_ids_config": TEST_PERSON_IDS,
    "train_persons": train_persons,
    "test_persons": test_persons,
    "train_recording_ids": train_recording_ids,
    "test_recording_ids": test_recording_ids,
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "train_class_distribution": encoded_class_counts(y_train),
    "test_class_distribution": encoded_class_counts(y_test),
    "person_overlap": person_overlap,
    "recording_overlap": recording_overlap,
    "dataset_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "dataset_sha256": dataset_sha256,
}
write_json(MODELS_DIR / "split_manifest.json", split_manifest)

print("Person-based holdout split da san sang.")
print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Train persons ({len(train_persons)}): {train_persons}")
print(f"Test persons ({len(test_persons)}): {test_persons}")
print(f"Person overlap: {person_overlap}")
print(f"Recording overlap: {recording_overlap}")
print_table("Train class distribution", pd.Series(encoded_class_counts(y_train)).rename_axis("label").reset_index(name="samples"))
print_table("Test class distribution", pd.Series(encoded_class_counts(y_test)).rename_axis("label").reset_index(name="samples"))
print(f"Saved split manifest: {MODELS_DIR / 'split_manifest.json'}")


Person-based holdout split da san sang.
Train samples: 3307
Test samples: 707
Train persons (11): ['person02', 'person03', 'person04', 'person05', 'person06', 'person07', 'person08', 'person09', 'person11', 'person13', 'person14']
Test persons (3): ['person01', 'person10', 'person12']
Person overlap: []
Recording overlap: []

Train class distribution
------------------------
         label  samples
       correct      879
forward_slouch      757
     lean_left      828
    lean_right      843

Test class distribution
-----------------------
         label  samples
       correct      172
forward_slouch      132
     lean_left      226
    lean_right      177
Saved split manifest: G:\PTIT TL\K1N3\PYTHON\BTL\smart_posture_monitor\models\split_manifest.json


In [5]:
unique_train_persons = groups_train.nunique()
if unique_train_persons < N_SPLITS:
    raise ValueError(
        f"Khong du train persons cho {N_SPLITS} folds: chi co {unique_train_persons} persons."
    )

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_rows = []
for fold_id, (fold_train_idx, fold_val_idx) in enumerate(cv.split(X_train, y_train, groups_train), start=1):
    fold_train_persons = set(groups_train.iloc[fold_train_idx])
    fold_val_persons = set(groups_train.iloc[fold_val_idx])
    fold_overlap = sorted(fold_train_persons & fold_val_persons)
    if fold_overlap:
        raise RuntimeError(f"Fold {fold_id} bi person leakage: {fold_overlap}")

    fold_rows.append({
        "fold": fold_id,
        "train_samples": int(len(fold_train_idx)),
        "val_samples": int(len(fold_val_idx)),
        "train_persons": len(fold_train_persons),
        "val_persons": len(fold_val_persons),
        "overlap_persons": fold_overlap,
        **{f"val_{label}": count for label, count in encoded_class_counts(y_train.iloc[fold_val_idx]).items()},
    })

fold_audit = pd.DataFrame(fold_rows)
print_table("StratifiedGroupKFold audit on train persons", fold_audit)



StratifiedGroupKFold audit on train persons
-------------------------------------------
 fold  train_samples  val_samples  train_persons  val_persons overlap_persons  val_correct  val_forward_slouch  val_lean_left  val_lean_right
    1           2285         1022              9            2              []          278                 209            273             262
    2           2809          498              9            2              []          124                 136            107             131
    3           2576          731              8            3              []          180                 163            197             191
    4           2788          519              9            2              []          145                 126            113             135
    5           2770          537              9            2              []          152                 123            138             124


In [6]:
SCORING = {
    "accuracy": "accuracy",
    "macro_precision": make_scorer(precision_score, average="macro", zero_division=0),
    "macro_recall": make_scorer(recall_score, average="macro", zero_division=0),
    "macro_f1": make_scorer(f1_score, average="macro", zero_division=0),
    "weighted_f1": make_scorer(f1_score, average="weighted", zero_division=0),
}

xgb_tree_method = "hist"
if USE_XGBOOST_GPU:
    xgb_tree_method = "hist"

models = {
    "DummyClassifier": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", random_state=RANDOM_STATE)),
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=None,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=None,
    ),
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=len(EXPECTED_CLASSES),
        eval_metric="mlogloss",
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method=xgb_tree_method,
    ),
}

print(f"Scoring metrics: {list(SCORING.keys())}")
print(f"Models: {list(models.keys())}")


Scoring metrics: ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1']
Models: ['DummyClassifier', 'LogisticRegression', 'SVM RBF', 'RandomForest', 'ExtraTrees', 'XGBoost']


In [7]:
baseline_rows = []
for model_name, estimator in models.items():
    print(f"Dang chay baseline CV: {model_name}")
    cv_scores = cross_validate(
        estimator,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
        return_train_score=False,
    )
    baseline_rows.append({
        "Model": model_name,
        "CV Accuracy Mean": float(np.mean(cv_scores["test_accuracy"])),
        "CV Macro Precision Mean": float(np.mean(cv_scores["test_macro_precision"])),
        "CV Macro Recall Mean": float(np.mean(cv_scores["test_macro_recall"])),
        "CV Macro F1 Mean": float(np.mean(cv_scores["test_macro_f1"])),
        "CV Macro F1 Std": float(np.std(cv_scores["test_macro_f1"], ddof=1)),
        "CV Weighted F1 Mean": float(np.mean(cv_scores["test_weighted_f1"])),
    })

baseline_results = (
    pd.DataFrame(baseline_rows)
    .sort_values("CV Macro F1 Mean", ascending=False)
    .reset_index(drop=True)
)
baseline_results.to_csv(RESULTS_DIR / "baseline_cv_results.csv", index=False)
print_table("Baseline CV results", baseline_results)
print(f"Saved baseline results: {RESULTS_DIR / 'baseline_cv_results.csv'}")


Dang chay baseline CV: DummyClassifier
Dang chay baseline CV: LogisticRegression
Dang chay baseline CV: SVM RBF
Dang chay baseline CV: RandomForest
Dang chay baseline CV: ExtraTrees
Dang chay baseline CV: XGBoost

Baseline CV results
-------------------
             Model  CV Accuracy Mean  CV Macro Precision Mean  CV Macro Recall Mean  CV Macro F1 Mean  CV Macro F1 Std  CV Weighted F1 Mean
           SVM RBF          0.586234                 0.666169              0.578957          0.560125         0.126592             0.561735
LogisticRegression          0.556540                 0.599466              0.548326          0.520498         0.166083             0.520778
        ExtraTrees          0.543784                 0.574104              0.537252          0.513266         0.145812             0.514467
           XGBoost          0.538487                 0.574567              0.531995          0.508276         0.132929             0.510126
      RandomForest          0.521225          

In [8]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", random_state=RANDOM_STATE)),
])

svm_param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.001, 0.01, 0.1],
    "model__class_weight": [None, "balanced"],
}

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

print("Dang tune SVM RBF bang GridSearchCV tren train persons...")
svm_search.fit(X_train, y_train, groups=groups_train)
svm_search_results = pd.DataFrame(svm_search.cv_results_)[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")
svm_search_results.to_csv(RESULTS_DIR / "svm_search_results.csv", index=False)

svm_best_std = float(svm_search.cv_results_["std_test_score"][svm_search.best_index_])
print(f"Best SVM params: {svm_search.best_params_}")
print(f"Best SVM CV Macro F1 mean: {svm_search.best_score_:.6f}")
print(f"Best SVM CV Macro F1 std: {svm_best_std:.6f}")
print(f"Saved SVM search results: {RESULTS_DIR / 'svm_search_results.csv'}")


Dang tune SVM RBF bang GridSearchCV tren train persons...
Best SVM params: {'model__C': 10, 'model__class_weight': None, 'model__gamma': 0.001}
Best SVM CV Macro F1 mean: 0.607130
Best SVM CV Macro F1 std: 0.170123
Saved SVM search results: G:\PTIT TL\K1N3\PYTHON\BTL\smart_posture_monitor\results\svm_search_results.csv


In [9]:
xgb_base = XGBClassifier(
    objective="multi:softprob",
    num_class=len(EXPECTED_CLASSES),
    eval_metric="mlogloss",
    random_state=RANDOM_STATE,
    n_jobs=1,
    tree_method=xgb_tree_method,
)

xgb_param_distributions = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.02, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1],
    "reg_alpha": [0, 0.1],
    "reg_lambda": [1, 5, 10],
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_distributions,
    n_iter=XGB_N_ITER,
    scoring="f1_macro",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

print("Dang tune XGBoost bang RandomizedSearchCV tren train persons...")
xgb_search.fit(X_train, y_train, groups=groups_train)
xgb_search_results = pd.DataFrame(xgb_search.cv_results_)[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")
xgb_search_results.to_csv(RESULTS_DIR / "xgboost_search_results.csv", index=False)

xgb_best_std = float(xgb_search.cv_results_["std_test_score"][xgb_search.best_index_])
print(f"Best XGBoost params: {xgb_search.best_params_}")
print(f"Best XGBoost CV Macro F1 mean: {xgb_search.best_score_:.6f}")
print(f"Best XGBoost CV Macro F1 std: {xgb_best_std:.6f}")
print(f"Saved XGBoost search results: {RESULTS_DIR / 'xgboost_search_results.csv'}")


Dang tune XGBoost bang RandomizedSearchCV tren train persons...
Best XGBoost params: {'subsample': 0.8, 'reg_lambda': 10, 'reg_alpha': 0.1, 'n_estimators': 600, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 1.0}
Best XGBoost CV Macro F1 mean: 0.518970
Best XGBoost CV Macro F1 std: 0.125608
Saved XGBoost search results: G:\PTIT TL\K1N3\PYTHON\BTL\smart_posture_monitor\results\xgboost_search_results.csv


In [10]:
selection_rows = []
for _, row in baseline_results.iterrows():
    selection_rows.append({
        "Model": row["Model"],
        "Source": "baseline_cv",
        "CV Macro F1 Mean": float(row["CV Macro F1 Mean"]),
        "CV Macro F1 Std": float(row["CV Macro F1 Std"]),
        "Best Params": "{}",
    })

selection_rows.extend([
    {
        "Model": "SVM RBF Tuned",
        "Source": "grid_search_cv",
        "CV Macro F1 Mean": float(svm_search.best_score_),
        "CV Macro F1 Std": svm_best_std,
        "Best Params": json.dumps(json_safe(svm_search.best_params_), sort_keys=True),
    },
    {
        "Model": "XGBoost Tuned",
        "Source": "randomized_search_cv",
        "CV Macro F1 Mean": float(xgb_search.best_score_),
        "CV Macro F1 Std": xgb_best_std,
        "Best Params": json.dumps(json_safe(xgb_search.best_params_), sort_keys=True),
    },
])

model_selection_results = (
    pd.DataFrame(selection_rows)
    .sort_values("CV Macro F1 Mean", ascending=False)
    .reset_index(drop=True)
)
model_selection_results.to_csv(RESULTS_DIR / "model_selection_results.csv", index=False)

best_row = model_selection_results.iloc[0].to_dict()
best_model_name = str(best_row["Model"])

if best_model_name == "SVM RBF Tuned":
    final_estimator = svm_search.best_estimator_
    best_params = svm_search.best_params_
elif best_model_name == "XGBoost Tuned":
    final_estimator = xgb_search.best_estimator_
    best_params = xgb_search.best_params_
else:
    final_estimator = clone(models[best_model_name])
    final_estimator.fit(X_train, y_train)
    best_params = {}

model_path = MODELS_DIR / "best_model.joblib"
joblib.dump(final_estimator, model_path)

training_metadata = {
    "model_name": best_model_name,
    "best_params": best_params,
    "cv_macro_f1_mean": float(best_row["CV Macro F1 Mean"]),
    "cv_macro_f1_std": float(best_row["CV Macro F1 Std"]),
    "feature_columns": feature_columns,
    "n_features": int(len(feature_columns)),
    "classes": list(EXPECTED_CLASSES),
    "label_to_id": LABEL_TO_ID,
    "id_to_label": {str(k): v for k, v in ID_TO_LABEL.items()},
    "random_state": RANDOM_STATE,
    "n_splits": N_SPLITS,
    "train_persons": train_persons,
    "test_persons": test_persons,
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "train_class_distribution": encoded_class_counts(y_train),
    "test_class_distribution": encoded_class_counts(y_test),
    "dataset_sha256": dataset_sha256,
    "dataset_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "training_timestamp": datetime.now(timezone.utc).isoformat(),
}
write_json(MODELS_DIR / "training_metadata.json", training_metadata)

print_table("Model selection results", model_selection_results)
print(f"Selected model: {best_model_name}")
print(f"Selected CV Macro F1 mean: {best_row['CV Macro F1 Mean']:.6f}")
print(f"Selected CV Macro F1 std: {best_row['CV Macro F1 Std']:.6f}")
print(f"Saved model: {model_path}")
print(f"Saved metadata: {MODELS_DIR / 'training_metadata.json'}")
print(f"Saved model selection table: {RESULTS_DIR / 'model_selection_results.csv'}")



Model selection results
-----------------------
             Model               Source  CV Macro F1 Mean  CV Macro F1 Std                                                                                                                                                                      Best Params
     SVM RBF Tuned       grid_search_cv          0.607130         0.170123                                                                                                             {"model__C": 10, "model__class_weight": null, "model__gamma": 0.001}
           SVM RBF          baseline_cv          0.560125         0.126592                                                                                                                                                                               {}
LogisticRegression          baseline_cv          0.520498         0.166083                                                                                                                         

In [11]:
loaded_model = joblib.load(MODELS_DIR / "best_model.joblib")
if not hasattr(loaded_model, "predict"):
    raise RuntimeError("Loaded model khong co method predict().")

# Smoke test chi dung TRAIN samples de tranh dung holdout test cho performance.
smoke_X = X_train.iloc[:5]
original_predictions = final_estimator.predict(smoke_X)
loaded_predictions = loaded_model.predict(smoke_X)
if not np.array_equal(original_predictions, loaded_predictions):
    raise RuntimeError("Prediction cua loaded model khong khop voi final estimator.")

if hasattr(loaded_model, "n_features_in_") and int(loaded_model.n_features_in_) != len(feature_columns):
    raise RuntimeError("So feature cua loaded model khong khop feature_columns.")

print("Reload smoke test PASS.")
print(f"Loaded model type: {type(loaded_model).__name__}")
print(f"Smoke test train predictions: {loaded_predictions.tolist()}")
print("Luu y: khong predict tren X_test trong notebook nay.")


Reload smoke test PASS.
Loaded model type: Pipeline
Smoke test train predictions: [0, 0, 0, 0, 0]
Luu y: khong predict tren X_test trong notebook nay.


# Final Conclusion

- Dataset: 4014 samples, 29 features, 14 persons, 16 recordings, 4 classes.
- Train split: 3307 samples from 11 persons: ['person02', 'person03', 'person04', 'person05', 'person06', 'person07', 'person08', 'person09', 'person11', 'person13', 'person14'].
- Test holdout split: 707 samples from 3 persons: ['person01', 'person10', 'person12'].
- Leakage checks: person overlap = []; recording overlap = [].
- Cross-validation: StratifiedGroupKFold with 5 folds on training persons only.
- Baseline CV results were saved to `results/baseline_cv_results.csv`.
- Best baseline by CV Macro F1: SVM RBF = 0.560125 +/- 0.126592.
- Best SVM params: `{'model__C': 10, 'model__class_weight': None, 'model__gamma': 0.001}` with CV Macro F1 = 0.607130 +/- 0.170123.
- Best XGBoost params: `{'subsample': 0.8, 'reg_lambda': 10, 'reg_alpha': 0.1, 'n_estimators': 600, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 1.0}` with CV Macro F1 = 0.518970 +/- 0.125608.
- Selected final model: SVM RBF Tuned with CV Macro F1 = 0.607130 +/- 0.170123.
- Saved artifacts: `models/best_model.joblib`, `models/split_manifest.json`, `models/training_metadata.json`.
- Saved result tables: `results/baseline_cv_results.csv`, `results/svm_search_results.csv`, `results/xgboost_search_results.csv`, `results/model_selection_results.csv`.

Test set has NOT been used for model performance evaluation. Final unseen-person evaluation will be performed in evaluate.py.
